# STEAM-GAME-PROJECT — Phân tích dữ liệu Non-ML

## Mục tiêu và phạm vi

Template cho **Phase 3.3 — Question Formulation** và **Phase 3.4 — Non-ML Data Analysis** của 5 câu hỏi Non-ML. Chưa thực hiện phân tích và chưa có kết quả nghiên cứu; phương pháp ở 3.3 là kế hoạch.

## Dataset

- **Chính:** Kaggle Steam Games, March 2025; tệp hiện có `data/raw/steam_games_mar2025.csv`.
- **Bổ sung:** `data/raw/hf_steam_games_dataset_train.parquet` từ Hugging Face; chỉ dùng khi thuộc tính có giá trị và khóa/snapshot/coverage đã xác minh.
- README và mặc định `src/data_loader.py` nêu `games_march2025_cleaned.csv`, khác tên CSV hiện có. Khi triển khai, truyền đường dẫn tương đối rõ ràng. `src/preprocessing.py` chưa có.
- Notebook 01 đã làm EDA chung. Tái sử dụng quy tắc phù hợp, không lặp toàn bộ EDA; xác minh sentinel, zero, unit và nguồn cho từng biến.

## Năm câu hỏi Non-ML chính thức

Lấy từ bảng phân công CONTRIBUTIONS: Q3 là câu hỏi Machine Learning, thuộc Notebook 03; Notebook này gồm Q1, Q2, Q4, Q5, Q6.

1. RQ1 (Q1 nhóm): Mở rộng Độ phủ & Quy mô Người chơi.
2. RQ2 (Q2 nhóm): Phân hóa Thời lượng chơi (Playtime Distribution).
3. RQ3 (Q4 nhóm): Giá trị Nội dung Bổ sung (DLC & Achievements).
4. RQ4 (Q5 nhóm): Động lực Tương tác F2P vs Pay-to-Play.
5. RQ5 (Q6 nhóm): Nghịch lý Thời lượng chơi.

Câu hỏi được trích nguyên văn; 3.3 là kế hoạch, 3.4 là chỗ triển khai sau này.

# 3.3. Question Formulation

Project brief yêu cầu mỗi câu hỏi có ý nghĩa và nêu lợi ích. D1–D6 dưới đây là kế hoạch, không phải kết quả đã hoàn thành; chỉ dùng tiêu chí phù hợp.

## 3.3.1. Research Question 1: Mở rộng Độ phủ & Quy mô Người chơi (Q1 nhóm)

### 1. Research Question

> “Số lượng ngôn ngữ và hệ điều hành được hỗ trợ có mối liên hệ như thế nào với lượng người sở hữu ước tính, và mối liên hệ này thay đổi ra sao khi xét đến thể loại và tuổi đời game?”

Association ở cấp game giữa độ phủ ngôn ngữ/nền tảng và owner ước tính, xem xét `genres`, tuổi đời; không suy luận nhân quả.

### 2. Motivation

Nhà phát triển/nhà phát hành cần hiểu độ phủ ngôn ngữ và nền tảng liên hệ thế nào với quy mô sở hữu theo phân khúc để ưu tiên thị trường, hỗ trợ nền tảng. Kết quả chỉ hỗ trợ quyết định và giả thuyết tiếp theo, không chứng minh quan hệ nhân quả.

**Stakeholders:** Nhà phát triển, nhà phát hành, đơn vị phân tích game và người chơi.

**Practical Value:** Ưu tiên nghiên cứu/phân khúc bản địa hóa và hỗ trợ nền tảng.

### 3. Attributes Used

| Attribute | Dataset Source | Role in Analysis |
| --- | --- | --- |
| appid | Kaggle CSV | Khóa game/ghép. |
| supported_languages | Kaggle CSV | Tạo `language_count`. |
| os | Kaggle CSV | Danh sách hệ điều hành để tạo số nền tảng. |
| estimated_owners | Kaggle CSV | Kết quả dạng khoảng. |
| genres, release_date, price | Kaggle CSV | Nhóm tuổi/biến điều chỉnh. |
| appID, windows, mac, linux | HF Parquet | Khóa/cờ đối chiếu; xác minh thời điểm trước ghép. |

### 4. Research Hypotheses

- H1: `language_count` và `platform_count` có association với `owner_midpoint`; không diễn giải nhân quả.
- H2: Mức association khác nhau theo `genres`.
- H3: Mức association thay đổi theo `game_age` sau khi xem xét `price` và `genres`.

### 5. Analytical Depth Criteria

Các phương pháp sau là kế hoạch dự kiến. Chỉ ghi nhận tiêu chí đã thực hiện khi có kiểm tra và kết quả tương ứng.

| Criterion | Planned Method | Purpose |
| --- | --- | --- |
| D1 — Combining Datasets | Đối chiếu `os` với cờ nền tảng HF sau kiểm tra khóa/snapshot/sai khác. | Đánh giá thông tin đối chiếu mới; chưa mặc định gộp. |
| D2 — Reshaping / Time | `explode()` ngôn ngữ; tính tuổi từ mốc snapshot đã xác nhận. | Tổng hợp ngôn ngữ/tuổi mà không nhân bản đơn vị kiểm định. |
| D3 — Derived Variables | `language_count`, `platform_count`, `owner_midpoint`, `game_age`. | Định nghĩa biến tái lập. |
| D4 — Controlling for a Confounder | Phân tầng/điều chỉnh `genres`, `price`, `game_age`. | Association có điều kiện, không loại mọi nhiễu. |
| D5 — Quantifying Uncertainty | Spearman và Bootstrap 95% CI theo game; báo effect size/N. | Định lượng bất định. |
| D6 — Robustness Check | Biên owner thấp/cao/trung điểm, nhóm tuổi, quy tắc outlier. | Độ nhạy với giả định. |

### 6. Research Limitations

Owner là khoảng; thuộc tính hỗ trợ là khai báo; genres có thể đa nhãn; dữ liệu quan sát không hỗ trợ nhân quả; snapshot/khóa cần xác minh.

## 3.3.2. Research Question 2: Phân hóa Thời lượng chơi (Playtime Distribution) (Q2 nhóm)

### 1. Research Question

> “Sự chênh lệch giữa thời gian chơi trung bình và trung vị phản ánh những đặc điểm khác biệt nào trong phân phối thời lượng chơi của các tựa game, và mức độ chênh lệch này có khác nhau giữa các thể loại game hay không?”

So sánh hai thống kê tổng hợp ghép cặp ở cấp game; không tái dựng toàn bộ phân phối người chơi.

### 2. Motivation

Mean nhạy với giá trị lớn hơn median. Đo gap theo thể loại giúp chọn cách tóm tắt engagement và nhóm cần khảo sát thêm; không mô tả từng người chơi.

**Stakeholders:** Nhà phát triển, nhà phát hành, phân tích sản phẩm và người chơi.

**Practical Value:** Định lượng hướng/độ lớn gap, khác biệt thể loại và uncertainty.

### 3. Attributes Used

| Attribute | Dataset Source | Role in Analysis |
| --- | --- | --- |
| appid, name | Kaggle CSV | Khóa/nhận diện. |
| average_playtime_forever, median_playtime_forever | Kaggle CSV | Hai số đo chính. |
| genres, estimated_owners, release_date | Kaggle CSV | Nhóm/biến điều chỉnh. |
| average_playtime_2weeks, median_playtime_2weeks | HF Parquet | Đối chiếu phụ; xác minh snapshot/độ phủ. |

### 4. Research Hypotheses

- H1: Trong game có hai số đo dương, khác nhau, tỷ lệ average > median khác 50%.
- H2: Tỷ lệ gap khác 0 và độ lớn gap tuyệt đối khác nhau theo genres.
- H3: Khác biệt thể loại còn sau khi xem xét owner và tuổi game.

### 5. Analytical Depth Criteria

Các phương pháp sau là kế hoạch dự kiến. Chỉ ghi nhận tiêu chí đã thực hiện khi có kiểm tra và kết quả tương ứng.

| Criterion | Planned Method | Purpose |
| --- | --- | --- |
| D1 — Combining Datasets | Ghép biến two-week nhưng giữ riêng source/window, không ghi đè forever. | Kiểm tra khác biệt theo cửa sổ sau khi xác minh snapshot. |
| D2 — Reshaping / Time | `melt()` mean/median; giữ cặp game và tách forever/two-week. | So sánh có cấu trúc, không coi dòng cùng game độc lập. |
| D3 — Derived Variables | Signed/absolute gap, ratio khi median>0, rank difference nếu phù hợp. | Phân biệt hướng/độ lớn; minh bạch undefined. |
| D4 — Controlling for a Confounder | Phân tầng/điều chỉnh genre, owner, age. | Giảm confounding quan sát được. |
| D5 — Quantifying Uncertainty | Bootstrap 95% CI theo game; sign/group test, effect size phù hợp. | Bất định có xét nhiều số 0. |
| D6 — Robustness Check | Toàn bộ so với cặp dương; gap/ratio/rank và outlier. | Độ nhạy zero-inflation/thước đo. |

### 6. Research Limitations

Playtime tổng hợp cấp game; tỷ lệ zero cao; unit/nguồn/cực trị cần xác minh; gap không mô tả đầy đủ phân phối.

## 3.3.3. Research Question 3: Giá trị Nội dung Bổ sung (DLC & Achievements) (Q4 nhóm)

### 1. Research Question

> “Mật độ phân bổ thành tựu trên mỗi gói mở rộng (achievements/dlc_count) có chiều hướng tác động như thế nào đến tỷ lệ đánh giá tích cực gần đây của người chơi (pct_pos_recent), sau khi đã kiểm soát yếu tố chất lượng chuyên môn của trò chơi (metacritic_score)?”

Association achievements trên DLC với pct_pos_recent, điều chỉnh metacritic_score; không nhân quả.

### 2. Motivation

Nhà phát triển có thể quan tâm liệu mật độ thành tựu liên hệ với positive review gần đây sau khi xét điểm chuyên môn, để đánh giá chỉ số thiết kế cần nghiên cứu. Dữ liệu quan sát chỉ hỗ trợ association, không chứng minh tác động.

**Stakeholders:** Nhà phát triển/nhà phát hành, thiết kế game và phân tích chất lượng.

**Practical Value:** Ước lượng association có điều kiện và đánh giá cách đo mật độ DLC/achievement.

### 3. Attributes Used

| Attribute | Dataset Source | Role in Analysis |
| --- | --- | --- |
| appid, achievements, dlc_count | Kaggle CSV | Khóa, tử số và mẫu số. |
| pct_pos_recent, num_reviews_recent | Kaggle CSV | Kết quả và độ tin cậy review. |
| appID, metacritic_score | HF Parquet | Khóa và biến kiểm soát bổ sung; kiểm tra ghép/snapshot. |

### 4. Research Hypotheses

- H1: Sau điều chỉnh metacritic_score, achievements-per-DLC liên hệ với pct_pos_recent.
- H2: Association có thể khác theo metacritic_score hoặc nhóm DLC.
- H3 (nếu phù hợp): Kết luận ổn định qua cách xử lý DLC=0 và ngưỡng review.

### 5. Analytical Depth Criteria

Các phương pháp sau là kế hoạch dự kiến. Chỉ ghi nhận tiêu chí đã thực hiện khi có kiểm tra và kết quả tương ứng.

| Criterion | Planned Method | Purpose |
| --- | --- | --- |
| D1 — Combining Datasets | Ghép metacritic_score HF; kiểm tra cardinality, snapshot, độ phủ/mã thiếu. | Bổ sung biến kiểm soát vắng ở CSV; chưa giả định hợp lệ. |
| D3 — Derived Variables | Tạo achievements/dlc_count; tách DLC=0, không chia 0. | Thước đo mật độ tái lập. |
| D4 — Controlling for a Confounder | Điều chỉnh metacritic_score; cân nhắc review count/genre. | Association có điều kiện. |
| D5 — Quantifying Uncertainty | Bootstrap CI 95% theo game và effect size phù hợp. | Định lượng bất định. |
| D6 — Robustness Check | Thử DLC=0, biến đổi/outlier và ngưỡng review. | Độ nhạy với định nghĩa và độ tin cậy review. |

### 6. Research Limitations

HF có thể khác snapshot; Metacritic có thể thiếu/mã đặc biệt; review phụ thuộc số lượng; ratio không đo trực tiếp chất lượng DLC; không suy luận nhân quả.

## 3.3.4. Research Question 4: Động lực Tương tác F2P vs Pay-to-Play (Q5 nhóm)

### 1. Research Question

> “Động lực Tương tác Người chơi giữa Mô hình Free-to-Play và Pay-to-Play: Liệu Game Miễn phí có duy trì được mức độ tham gia tương đối (Relative Recent Activity) cao hơn Game Trả phí sau khi kiểm soát các tựa game Outlier ngoại lai (Top 1% Blockbuster)?”

So sánh proxy hoạt động gần đây giữa F2P/paid sau xử lý top 1%; định nghĩa Relative Recent Activity cần xác nhận.

### 2. Motivation

Nhà phát hành cần so sánh hoạt động gần đây giữa mô hình giá mà không để vài game blockbuster chi phối, hỗ trợ quyết định phân khúc. Cần xác định trước metric hoạt động.

**Stakeholders:** Nhà phát hành, nhà phát triển và phân tích thị trường.

**Practical Value:** So sánh F2P/paid sau quy tắc top 1% minh bạch; không khẳng định mô hình giá gây engagement.

### 3. Attributes Used

| Attribute | Dataset Source | Role in Analysis |
| --- | --- | --- |
| appid, price | Kaggle CSV | Khóa và nhóm F2P/paid. |
| estimated_owners, peak_ccu, genres, release_date | Kaggle CSV | Quy mô/ứng viên confounder; peak_ccu không mặc định là recent activity. |
| average_playtime_2weeks | HF Parquet | Ứng viên proxy recent activity; xác minh unit/snapshot/coverage. |

### 4. Research Hypotheses

- H1: Theo metric và top-1% rule xác định trước, F2P có Relative Recent Activity cao hơn paid.
- H2: Khác biệt còn sau xem xét genre, game_age và quy mô owner.
- H3 (nếu phù hợp): Kết luận nhạy với ngưỡng blockbuster hoặc activity proxy.

### 5. Analytical Depth Criteria

Các phương pháp sau là kế hoạch dự kiến. Chỉ ghi nhận tiêu chí đã thực hiện khi có kiểm tra và kết quả tương ứng.

| Criterion | Planned Method | Purpose |
| --- | --- | --- |
| D1 — Combining Datasets | Thêm average_playtime_2weeks HF nếu chọn; giữ rõ source/window, kiểm tra ID/snapshot/coverage. | Bổ sung proxy gần đây không có trong CSV chính. |
| D3 — Derived Variables | Tạo is_free, relative_recent_activity, top-1% flag theo quy tắc định trước. | Nhóm và outlier tái lập. |
| D4 — Controlling for a Confounder | Điều chỉnh genre, game_age, owner và biến liên quan. | So sánh có điều kiện. |
| D5 — Quantifying Uncertainty | Bootstrap CI 95% chênh nhóm/effect size. | Định lượng bất định. |
| D6 — Robustness Check | Thử ngưỡng 1%, proxy activity, giá/zero coding; có-không loại top outliers. | Độ nhạy đặc tả. |

### 6. Research Limitations

Relative Recent Activity cần operationalize; HF có thể thưa/khác snapshot; peak_ccu không tương đương activity gần đây; giá hiện tại khác giá lúc phát hành; association không nhân quả.

## 3.3.5. Research Question 5: Nghịch lý Thời lượng chơi (The Playtime Paradox) (Q6 nhóm)

### 1. Research Question

> “Nghịch lý thời lượng chơi (The Playtime Paradox): Game có thời lượng chơi càng dài có thực sự mang lại mức độ hài lòng cao hơn cho game thủ, hay việc cố tình kéo dài nội dung (Content Bloating) đang gây ra hiệu ứng mệt mỏi phản tác dụng?”

Association giữa playtime và proxy đánh giá; dataset không quan sát ý định kéo dài nội dung hay mệt mỏi cá nhân.

### 2. Motivation

Nhà thiết kế cần hiểu quan hệ playtime và phản hồi tích cực, nhưng duration dài không đồng nghĩa hài lòng. Phân tích xu hướng/phi tuyến có thể gợi hướng khảo sát thiết kế tiếp, không xác nhận cơ chế bloating.

**Stakeholders:** Nhà thiết kế, nhà phát triển, nhà phát hành và người chơi.

**Practical Value:** Kiểm tra phi tuyến/bão hòa trong association playtime–review proxy; không kết luận causal bloating.

### 3. Attributes Used

| Attribute | Dataset Source | Role in Analysis |
| --- | --- | --- |
| appid, average_playtime_forever, median_playtime_forever | Kaggle CSV | Khóa và số đo duration. |
| pct_pos_total, pct_pos_recent, num_reviews_total, num_reviews_recent | Kaggle CSV | Proxy satisfaction/review reliability. |
| genres, release_date, estimated_owners | Kaggle CSV | Ứng viên confounder. |
| average_playtime_2weeks, median_playtime_2weeks | HF Parquet | Đối chiếu recent/lifetime nếu snapshot tương thích. |

### 4. Research Hypotheses

- H1: Association duration–review proxy có thể phi tuyến; kiểm tra fit với tuyến tính.
- H2: Có thể có bão hòa/giảm ở duration cao; diễn giải pattern, không suy ra bloating.
- H3: Dạng association thay đổi sau xét genre, age, owner, review count.

### 5. Analytical Depth Criteria

Các phương pháp sau là kế hoạch dự kiến. Chỉ ghi nhận tiêu chí đã thực hiện khi có kiểm tra và kết quả tương ứng.

| Criterion | Planned Method | Purpose |
| --- | --- | --- |
| D1 — Combining Datasets | Thêm HF two-week nếu hợp lệ; giữ cửa sổ/source riêng, xác minh snapshot/coverage. | Đối chiếu recent/lifetime. |
| D2 — Reshaping / Time | Reshape mean/median hoặc lifetime/recent; giữ game pairing và phân biệt window. | So sánh loại số đo/cửa sổ. |
| D3 — Derived Variables | Log1p/spline duration; gap/ratio hợp lệ; định nghĩa điểm bão hòa. | Mô tả phi tuyến tái lập. |
| D4 — Controlling for a Confounder | Điều chỉnh genre, age, owners, review counts. | Giảm confounding quan sát. |
| D5 — Quantifying Uncertainty | Bootstrap CI 95% trend/turning point; effect size, support range. | Bất định nhất là vùng đuôi. |
| D6 — Robustness Check | Thay proxy, spline/linear, zero/outlier, review threshold, source/window. | Độ ổn định pattern. |

### 6. Research Limitations

Positive review chỉ là proxy; không có dữ liệu cá nhân, session, độ dài nội dung, ý định hay fatigue. Survivorship, age, review volume, genre và snapshot gây nhiễu; không kết luận bloating gây phản tác dụng.

# 3.4. Non-ML Data Analysis

Mỗi câu hỏi có pipeline riêng. Code cells chỉ là skeleton comment; không chạy phân tích hay tạo kết quả giả. Chạy từ project root; xác nhận đường dẫn, dependencies, filters và lựa chọn còn mở trước khi chạy.

## 3.4.1. Analysis of Research Question 1: Mở rộng Độ phủ & Quy mô Người chơi (Q1 nhóm)

### 1. Research Objective

Ước lượng association độ phủ với owner và so sánh theo thể loại/tuổi.

### 2. Data Preparation

Kaggle CSV là nguồn chính; HF chỉ đối chiếu nền tảng sau xác minh D1. Kiểm tra khóa, danh sách, owner dạng khoảng và ngày. Không lặp EDA chung.

- **Dataset chính:** `data/raw/steam_games_mar2025.csv`.
- **Loader:** ưu tiên `src.data_loader.load_data(filepath="data/raw/steam_games_mar2025.csv")`; không dùng mặc định đang trỏ tới tên khác.
- **Dataset bổ sung:** `data/raw/hf_steam_games_dataset_train.parquet` chỉ khi D1 phù hợp.
- Tái sử dụng preprocessing Notebook 01; ghi filters, exclusions, unit, source, sample size và lý do. Không thay sentinel/zero nếu chưa xác minh.

### 3. Data Preprocessing

Hoàn thiện câu hỏi này; không lặp preprocessing tổng quát của Notebook 01.

In [ ]:
# Load required data
# TODO: chạy từ project root; dùng src.data_loader.load_data(filepath="data/raw/steam_games_mar2025.csv")
# TODO: chỉ load Parquet từ data/raw nếu D1 của câu hỏi này cần và đã xác minh.

# Select relevant attributes
# TODO: chọn đúng các cột trong Attributes Used.

# Handle missing values
# TODO: phân biệt NaN, sentinel và zero theo tài liệu nguồn.

# Apply filtering conditions
# TODO: ghi điều kiện lọc, ngưỡng, số dòng giữ/loại.

# Create derived variables
# TODO: tạo biến đã định nghĩa ở 3.3, xử lý trường hợp biên.

# Validate processed data
# TODO: kiểm tra types, ranges, keys, row counts và logic biến dẫn xuất.

### 4. Exploratory Analysis

Chỉ triển khai nội dung phù hợp; ghi cỡ mẫu và diễn giải có căn cứ.

In [ ]:
# Descriptive statistics
# TODO: thống kê mô tả cho các biến và tập phân tích.

# Distribution analysis
# TODO: kiểm tra phân phối, zero, sentinel, outlier phù hợp.

# Group comparison
# TODO: so sánh nhóm liên quan; ghi N của từng nhóm.

# Correlation analysis (if appropriate)
# TODO: chỉ tính nếu thang đo và đơn vị quan sát phù hợp.

### 5. Statistical Analysis

Chọn test theo estimand, cấu trúc dữ liệu và giả định; báo effect size cùng uncertainty.

In [ ]:
# Primary analytical method
# TODO: triển khai phương pháp chính đã định trước.

# Statistical tests (if appropriate)
# TODO: ghi test, giả thuyết và giả định.

# Confidence intervals / Bootstrap (if appropriate)
# TODO: tính CI theo đúng đơn vị lấy mẫu.

# Effect size (if appropriate)
# TODO: báo effect size phù hợp, không chỉ p-value.

### 6. Visualization

#### Visualization 1: Độ phủ và owner

- **Mục tiêu:** Quan sát quan hệ giữa số ngôn ngữ/nền tảng và owner.
- **Loại biểu đồ dự kiến:** Scatter/hexbin; thang log nếu phù hợp.
- **Trạng thái:** chưa triển khai.

In [ ]:
# TODO: tạo biểu đồ "Độ phủ và owner".
# Kiểm tra nhãn, đơn vị, cỡ mẫu, thang đo và chú giải.

**Nhận xét:** [Điền nhận xét dựa trên số liệu sau khi tạo biểu đồ.]

#### Visualization 2: Association theo nhóm

- **Mục tiêu:** So sánh liên hệ theo genre/tuổi.
- **Loại biểu đồ dự kiến:** Forest plot hoặc phân phối theo nhóm.
- **Trạng thái:** chưa triển khai.

In [ ]:
# TODO: tạo biểu đồ "Association theo nhóm".
# Kiểm tra nhãn, đơn vị, cỡ mẫu, thang đo và chú giải.

**Nhận xét:** [Điền nhận xét dựa trên số liệu sau khi tạo biểu đồ.]

### 7. Results and Discussion

- **Phát hiện chính:** [Điền số liệu sau phân tích.]
- **Diễn giải:** [Giải thích trong phạm vi dữ liệu/phương pháp.]
- **Hypotheses:** [Đối chiếu từng giả thuyết, nêu uncertainty và bằng chứng.]
- **Trạng thái:** chưa có kết quả.

### 8. Robustness Check

- **Phương pháp dự kiến:** Đối chiếu owner biên thấp/cao/trung điểm, cách chia game_age, quy tắc outlier và đối chiếu HF nếu tương thích.
- **Kết quả/nhận xét:** [Điền sau khi chạy; nêu đặc tả, N và mức thay đổi.]

### 9. Limitations

[Điền giới hạn phát sinh từ dữ liệu, phép đo, chọn mẫu, giả định và uncertainty.]

### 10. Conclusion

[Trả lời trực tiếp câu hỏi bằng kết quả đã kiểm chứng, nêu uncertainty và giới hạn suy luận. Để trống đến khi phân tích.]

## 3.4.2. Analysis of Research Question 2: Phân hóa Thời lượng chơi (Playtime Distribution) (Q2 nhóm)

### 1. Research Objective

Mô tả hướng/tỷ lệ/độ lớn gap mean–median và so sánh thể loại.

### 2. Data Preparation

Kaggle forever là phân tích chính. Notebook 01 ghi nhận nhiều số 0; không xem 0 là thiếu mặc định. HF two-week chỉ đối chiếu sau xác minh thời điểm, unit và độ phủ.

- **Dataset chính:** `data/raw/steam_games_mar2025.csv`.
- **Loader:** ưu tiên `src.data_loader.load_data(filepath="data/raw/steam_games_mar2025.csv")`; không dùng mặc định đang trỏ tới tên khác.
- **Dataset bổ sung:** `data/raw/hf_steam_games_dataset_train.parquet` chỉ khi D1 phù hợp.
- Tái sử dụng preprocessing Notebook 01; ghi filters, exclusions, unit, source, sample size và lý do. Không thay sentinel/zero nếu chưa xác minh.

### 3. Data Preprocessing

Hoàn thiện câu hỏi này; không lặp preprocessing tổng quát của Notebook 01.

In [ ]:
# Load required data
# TODO: chạy từ project root; dùng src.data_loader.load_data(filepath="data/raw/steam_games_mar2025.csv")
# TODO: chỉ load Parquet từ data/raw nếu D1 của câu hỏi này cần và đã xác minh.

# Select relevant attributes
# TODO: chọn đúng các cột trong Attributes Used.

# Handle missing values
# TODO: phân biệt NaN, sentinel và zero theo tài liệu nguồn.

# Apply filtering conditions
# TODO: ghi điều kiện lọc, ngưỡng, số dòng giữ/loại.

# Create derived variables
# TODO: tạo biến đã định nghĩa ở 3.3, xử lý trường hợp biên.

# Validate processed data
# TODO: kiểm tra types, ranges, keys, row counts và logic biến dẫn xuất.

### 4. Exploratory Analysis

Chỉ triển khai nội dung phù hợp; ghi cỡ mẫu và diễn giải có căn cứ.

In [ ]:
# Descriptive statistics
# TODO: thống kê mô tả cho các biến và tập phân tích.

# Distribution analysis
# TODO: kiểm tra phân phối, zero, sentinel, outlier phù hợp.

# Group comparison
# TODO: so sánh nhóm liên quan; ghi N của từng nhóm.

# Correlation analysis (if appropriate)
# TODO: chỉ tính nếu thang đo và đơn vị quan sát phù hợp.

### 5. Statistical Analysis

Chọn test theo estimand, cấu trúc dữ liệu và giả định; báo effect size cùng uncertainty.

In [ ]:
# Primary analytical method
# TODO: triển khai phương pháp chính đã định trước.

# Statistical tests (if appropriate)
# TODO: ghi test, giả thuyết và giả định.

# Confidence intervals / Bootstrap (if appropriate)
# TODO: tính CI theo đúng đơn vị lấy mẫu.

# Effect size (if appropriate)
# TODO: báo effect size phù hợp, không chỉ p-value.

### 6. Visualization

#### Visualization 1: Phân phối gap

- **Mục tiêu:** Quan sát hướng, magnitude, khối lượng tại 0.
- **Loại biểu đồ dự kiến:** Histogram/ECDF.
- **Trạng thái:** chưa triển khai.

In [ ]:
# TODO: tạo biểu đồ "Phân phối gap".
# Kiểm tra nhãn, đơn vị, cỡ mẫu, thang đo và chú giải.

**Nhận xét:** [Điền nhận xét dựa trên số liệu sau khi tạo biểu đồ.]

#### Visualization 2: Gap theo genre

- **Mục tiêu:** So sánh prevalence/magnitude.
- **Loại biểu đồ dự kiến:** Box/violin/point CI.
- **Trạng thái:** chưa triển khai.

In [ ]:
# TODO: tạo biểu đồ "Gap theo genre".
# Kiểm tra nhãn, đơn vị, cỡ mẫu, thang đo và chú giải.

**Nhận xét:** [Điền nhận xét dựa trên số liệu sau khi tạo biểu đồ.]

### 7. Results and Discussion

- **Phát hiện chính:** [Điền số liệu sau phân tích.]
- **Diễn giải:** [Giải thích trong phạm vi dữ liệu/phương pháp.]
- **Hypotheses:** [Đối chiếu từng giả thuyết, nêu uncertainty và bằng chứng.]
- **Trạng thái:** chưa có kết quả.

### 8. Robustness Check

- **Phương pháp dự kiến:** So sánh toàn mẫu/tập dương; signed/absolute gap, ratio, rank; xử lý outlier robust; không tự loại zero.
- **Kết quả/nhận xét:** [Điền sau khi chạy; nêu đặc tả, N và mức thay đổi.]

### 9. Limitations

[Điền giới hạn phát sinh từ dữ liệu, phép đo, chọn mẫu, giả định và uncertainty.]

### 10. Conclusion

[Trả lời trực tiếp câu hỏi bằng kết quả đã kiểm chứng, nêu uncertainty và giới hạn suy luận. Để trống đến khi phân tích.]

## 3.4.3. Analysis of Research Question 3: Giá trị Nội dung Bổ sung (DLC & Achievements) (Q4 nhóm)

### 1. Research Objective

Đánh giá association giữa mật độ achievements/DLC và positive review gần đây có kiểm soát điểm chuyên môn.

### 2. Data Preparation

Kaggle có achievements, dlc_count, pct_pos_recent, num_reviews_recent; metacritic_score dự kiến từ HF. Kiểm tra khóa, sentinel và số review trước dùng.

- **Dataset chính:** `data/raw/steam_games_mar2025.csv`.
- **Loader:** ưu tiên `src.data_loader.load_data(filepath="data/raw/steam_games_mar2025.csv")`; không dùng mặc định đang trỏ tới tên khác.
- **Dataset bổ sung:** `data/raw/hf_steam_games_dataset_train.parquet` chỉ khi D1 phù hợp.
- Tái sử dụng preprocessing Notebook 01; ghi filters, exclusions, unit, source, sample size và lý do. Không thay sentinel/zero nếu chưa xác minh.

### 3. Data Preprocessing

Hoàn thiện câu hỏi này; không lặp preprocessing tổng quát của Notebook 01.

In [ ]:
# Load required data
# TODO: chạy từ project root; dùng src.data_loader.load_data(filepath="data/raw/steam_games_mar2025.csv")
# TODO: chỉ load Parquet từ data/raw nếu D1 của câu hỏi này cần và đã xác minh.

# Select relevant attributes
# TODO: chọn đúng các cột trong Attributes Used.

# Handle missing values
# TODO: phân biệt NaN, sentinel và zero theo tài liệu nguồn.

# Apply filtering conditions
# TODO: ghi điều kiện lọc, ngưỡng, số dòng giữ/loại.

# Create derived variables
# TODO: tạo biến đã định nghĩa ở 3.3, xử lý trường hợp biên.

# Validate processed data
# TODO: kiểm tra types, ranges, keys, row counts và logic biến dẫn xuất.

### 4. Exploratory Analysis

Chỉ triển khai nội dung phù hợp; ghi cỡ mẫu và diễn giải có căn cứ.

In [ ]:
# Descriptive statistics
# TODO: thống kê mô tả cho các biến và tập phân tích.

# Distribution analysis
# TODO: kiểm tra phân phối, zero, sentinel, outlier phù hợp.

# Group comparison
# TODO: so sánh nhóm liên quan; ghi N của từng nhóm.

# Correlation analysis (if appropriate)
# TODO: chỉ tính nếu thang đo và đơn vị quan sát phù hợp.

### 5. Statistical Analysis

Chọn test theo estimand, cấu trúc dữ liệu và giả định; báo effect size cùng uncertainty.

In [ ]:
# Primary analytical method
# TODO: triển khai phương pháp chính đã định trước.

# Statistical tests (if appropriate)
# TODO: ghi test, giả thuyết và giả định.

# Confidence intervals / Bootstrap (if appropriate)
# TODO: tính CI theo đúng đơn vị lấy mẫu.

# Effect size (if appropriate)
# TODO: báo effect size phù hợp, không chỉ p-value.

### 6. Visualization

#### Visualization 1: Mật độ và review gần đây

- **Mục tiêu:** Quan sát association/phi tuyến.
- **Loại biểu đồ dự kiến:** Scatter/hexbin + trend.
- **Trạng thái:** chưa triển khai.

In [ ]:
# TODO: tạo biểu đồ "Mật độ và review gần đây".
# Kiểm tra nhãn, đơn vị, cỡ mẫu, thang đo và chú giải.

**Nhận xét:** [Điền nhận xét dựa trên số liệu sau khi tạo biểu đồ.]

#### Visualization 2: Association theo Metacritic

- **Mục tiêu:** So sánh các mức điểm chuyên môn.
- **Loại biểu đồ dự kiến:** Faceted plot/coefficient plot.
- **Trạng thái:** chưa triển khai.

In [ ]:
# TODO: tạo biểu đồ "Association theo Metacritic".
# Kiểm tra nhãn, đơn vị, cỡ mẫu, thang đo và chú giải.

**Nhận xét:** [Điền nhận xét dựa trên số liệu sau khi tạo biểu đồ.]

### 7. Results and Discussion

- **Phát hiện chính:** [Điền số liệu sau phân tích.]
- **Diễn giải:** [Giải thích trong phạm vi dữ liệu/phương pháp.]
- **Hypotheses:** [Đối chiếu từng giả thuyết, nêu uncertainty và bằng chứng.]
- **Trạng thái:** chưa có kết quả.

### 8. Robustness Check

- **Phương pháp dự kiến:** Đối chiếu DLC=0, biến đổi robust, ngưỡng num_reviews_recent và dùng/không dùng Metacritic sau kiểm tra.
- **Kết quả/nhận xét:** [Điền sau khi chạy; nêu đặc tả, N và mức thay đổi.]

### 9. Limitations

[Điền giới hạn phát sinh từ dữ liệu, phép đo, chọn mẫu, giả định và uncertainty.]

### 10. Conclusion

[Trả lời trực tiếp câu hỏi bằng kết quả đã kiểm chứng, nêu uncertainty và giới hạn suy luận. Để trống đến khi phân tích.]

## 3.4.4. Analysis of Research Question 4: Động lực Tương tác F2P vs Pay-to-Play (Q5 nhóm)

### 1. Research Objective

So sánh F2P/paid trên proxy activity đã định nghĩa, có xét outlier và confounders.

### 2. Data Preparation

Xác định Relative Recent Activity trước phân tích. average_playtime_2weeks chỉ ở HF; peak_ccu là concurrent peak, không đồng nghĩa recent activity. Xác minh giá F2P, snapshot, unit, top1%.

- **Dataset chính:** `data/raw/steam_games_mar2025.csv`.
- **Loader:** ưu tiên `src.data_loader.load_data(filepath="data/raw/steam_games_mar2025.csv")`; không dùng mặc định đang trỏ tới tên khác.
- **Dataset bổ sung:** `data/raw/hf_steam_games_dataset_train.parquet` chỉ khi D1 phù hợp.
- Tái sử dụng preprocessing Notebook 01; ghi filters, exclusions, unit, source, sample size và lý do. Không thay sentinel/zero nếu chưa xác minh.

### 3. Data Preprocessing

Hoàn thiện câu hỏi này; không lặp preprocessing tổng quát của Notebook 01.

In [ ]:
# Load required data
# TODO: chạy từ project root; dùng src.data_loader.load_data(filepath="data/raw/steam_games_mar2025.csv")
# TODO: chỉ load Parquet từ data/raw nếu D1 của câu hỏi này cần và đã xác minh.

# Select relevant attributes
# TODO: chọn đúng các cột trong Attributes Used.

# Handle missing values
# TODO: phân biệt NaN, sentinel và zero theo tài liệu nguồn.

# Apply filtering conditions
# TODO: ghi điều kiện lọc, ngưỡng, số dòng giữ/loại.

# Create derived variables
# TODO: tạo biến đã định nghĩa ở 3.3, xử lý trường hợp biên.

# Validate processed data
# TODO: kiểm tra types, ranges, keys, row counts và logic biến dẫn xuất.

### 4. Exploratory Analysis

Chỉ triển khai nội dung phù hợp; ghi cỡ mẫu và diễn giải có căn cứ.

In [ ]:
# Descriptive statistics
# TODO: thống kê mô tả cho các biến và tập phân tích.

# Distribution analysis
# TODO: kiểm tra phân phối, zero, sentinel, outlier phù hợp.

# Group comparison
# TODO: so sánh nhóm liên quan; ghi N của từng nhóm.

# Correlation analysis (if appropriate)
# TODO: chỉ tính nếu thang đo và đơn vị quan sát phù hợp.

### 5. Statistical Analysis

Chọn test theo estimand, cấu trúc dữ liệu và giả định; báo effect size cùng uncertainty.

In [ ]:
# Primary analytical method
# TODO: triển khai phương pháp chính đã định trước.

# Statistical tests (if appropriate)
# TODO: ghi test, giả thuyết và giả định.

# Confidence intervals / Bootstrap (if appropriate)
# TODO: tính CI theo đúng đơn vị lấy mẫu.

# Effect size (if appropriate)
# TODO: báo effect size phù hợp, không chỉ p-value.

### 6. Visualization

#### Visualization 1: Activity theo mô hình giá

- **Mục tiêu:** So sánh F2P/paid và outlier.
- **Loại biểu đồ dự kiến:** Violin/ECDF/box.
- **Trạng thái:** chưa triển khai.

In [ ]:
# TODO: tạo biểu đồ "Activity theo mô hình giá".
# Kiểm tra nhãn, đơn vị, cỡ mẫu, thang đo và chú giải.

**Nhận xét:** [Điền nhận xét dựa trên số liệu sau khi tạo biểu đồ.]

#### Visualization 2: Trước/sau top1%

- **Mục tiêu:** Minh họa ảnh hưởng blockbuster.
- **Loại biểu đồ dự kiến:** Hai-panel difference/CI plot.
- **Trạng thái:** chưa triển khai.

In [ ]:
# TODO: tạo biểu đồ "Trước/sau top1%".
# Kiểm tra nhãn, đơn vị, cỡ mẫu, thang đo và chú giải.

**Nhận xét:** [Điền nhận xét dựa trên số liệu sau khi tạo biểu đồ.]

### 7. Results and Discussion

- **Phát hiện chính:** [Điền số liệu sau phân tích.]
- **Diễn giải:** [Giải thích trong phạm vi dữ liệu/phương pháp.]
- **Hypotheses:** [Đối chiếu từng giả thuyết, nêu uncertainty và bằng chứng.]
- **Trạng thái:** chưa có kết quả.

### 8. Robustness Check

- **Phương pháp dự kiến:** Thử activity proxy, mẫu số chuẩn hóa, threshold top1%, cách mã hóa giá và inclusion outlier.
- **Kết quả/nhận xét:** [Điền sau khi chạy; nêu đặc tả, N và mức thay đổi.]

### 9. Limitations

[Điền giới hạn phát sinh từ dữ liệu, phép đo, chọn mẫu, giả định và uncertainty.]

### 10. Conclusion

[Trả lời trực tiếp câu hỏi bằng kết quả đã kiểm chứng, nêu uncertainty và giới hạn suy luận. Để trống đến khi phân tích.]

## 3.4.5. Analysis of Research Question 5: Nghịch lý Thời lượng chơi (The Playtime Paradox) (Q6 nhóm)

### 1. Research Objective

Mô tả association playtime–positive review, kiểm tra phi tuyến/bão hòa và độ ổn định sau điều chỉnh.

### 2. Data Preparation

Kaggle playtime/review là chính; xác minh unit, sentinel, zero và số review. HF two-week là sensitivity sau kiểm tra snapshot/ghép. Không lặp EDA chung.

- **Dataset chính:** `data/raw/steam_games_mar2025.csv`.
- **Loader:** ưu tiên `src.data_loader.load_data(filepath="data/raw/steam_games_mar2025.csv")`; không dùng mặc định đang trỏ tới tên khác.
- **Dataset bổ sung:** `data/raw/hf_steam_games_dataset_train.parquet` chỉ khi D1 phù hợp.
- Tái sử dụng preprocessing Notebook 01; ghi filters, exclusions, unit, source, sample size và lý do. Không thay sentinel/zero nếu chưa xác minh.

### 3. Data Preprocessing

Hoàn thiện câu hỏi này; không lặp preprocessing tổng quát của Notebook 01.

In [ ]:
# Load required data
# TODO: chạy từ project root; dùng src.data_loader.load_data(filepath="data/raw/steam_games_mar2025.csv")
# TODO: chỉ load Parquet từ data/raw nếu D1 của câu hỏi này cần và đã xác minh.

# Select relevant attributes
# TODO: chọn đúng các cột trong Attributes Used.

# Handle missing values
# TODO: phân biệt NaN, sentinel và zero theo tài liệu nguồn.

# Apply filtering conditions
# TODO: ghi điều kiện lọc, ngưỡng, số dòng giữ/loại.

# Create derived variables
# TODO: tạo biến đã định nghĩa ở 3.3, xử lý trường hợp biên.

# Validate processed data
# TODO: kiểm tra types, ranges, keys, row counts và logic biến dẫn xuất.

### 4. Exploratory Analysis

Chỉ triển khai nội dung phù hợp; ghi cỡ mẫu và diễn giải có căn cứ.

In [ ]:
# Descriptive statistics
# TODO: thống kê mô tả cho các biến và tập phân tích.

# Distribution analysis
# TODO: kiểm tra phân phối, zero, sentinel, outlier phù hợp.

# Group comparison
# TODO: so sánh nhóm liên quan; ghi N của từng nhóm.

# Correlation analysis (if appropriate)
# TODO: chỉ tính nếu thang đo và đơn vị quan sát phù hợp.

### 5. Statistical Analysis

Chọn test theo estimand, cấu trúc dữ liệu và giả định; báo effect size cùng uncertainty.

In [ ]:
# Primary analytical method
# TODO: triển khai phương pháp chính đã định trước.

# Statistical tests (if appropriate)
# TODO: ghi test, giả thuyết và giả định.

# Confidence intervals / Bootstrap (if appropriate)
# TODO: tính CI theo đúng đơn vị lấy mẫu.

# Effect size (if appropriate)
# TODO: báo effect size phù hợp, không chỉ p-value.

### 6. Visualization

#### Visualization 1: Playtime và review proxy

- **Mục tiêu:** Quan sát trend, density, vùng dữ liệu.
- **Loại biểu đồ dự kiến:** Hexbin/scatter + smoother/CI.
- **Trạng thái:** chưa triển khai.

In [ ]:
# TODO: tạo biểu đồ "Playtime và review proxy".
# Kiểm tra nhãn, đơn vị, cỡ mẫu, thang đo và chú giải.

**Nhận xét:** [Điền nhận xét dựa trên số liệu sau khi tạo biểu đồ.]

#### Visualization 2: Trend theo nhóm

- **Mục tiêu:** So sánh pattern/saturation.
- **Loại biểu đồ dự kiến:** Facet trend hoặc curve CI.
- **Trạng thái:** chưa triển khai.

In [ ]:
# TODO: tạo biểu đồ "Trend theo nhóm".
# Kiểm tra nhãn, đơn vị, cỡ mẫu, thang đo và chú giải.

**Nhận xét:** [Điền nhận xét dựa trên số liệu sau khi tạo biểu đồ.]

### 7. Results and Discussion

- **Phát hiện chính:** [Điền số liệu sau phân tích.]
- **Diễn giải:** [Giải thích trong phạm vi dữ liệu/phương pháp.]
- **Hypotheses:** [Đối chiếu từng giả thuyết, nêu uncertainty và bằng chứng.]
- **Trạng thái:** chưa có kết quả.

### 8. Robustness Check

- **Phương pháp dự kiến:** Đối chiếu total/recent review, average/median, linear/spline, zero/outlier, review threshold và HF two-week nếu đủ tương thích.
- **Kết quả/nhận xét:** [Điền sau khi chạy; nêu đặc tả, N và mức thay đổi.]

### 9. Limitations

[Điền giới hạn phát sinh từ dữ liệu, phép đo, chọn mẫu, giả định và uncertainty.]

### 10. Conclusion

[Trả lời trực tiếp câu hỏi bằng kết quả đã kiểm chứng, nêu uncertainty và giới hạn suy luận. Để trống đến khi phân tích.]